# PII Detection with vLLM on Databricks

Run open-weight LLMs on a single-node GPU cluster to detect and mask personally identifiable information in financial documents. The notebook is parameterized by `model_id` so the same code runs for any HuggingFace model; a Databricks Asset Bundle job chains multiple models sequentially on the same cluster.

**Capabilities demonstrated:**
- vLLM structured (JSON-schema-constrained) generation via guided decoding
- System-prompt prefix caching for amortized KV computation
- GPU memory and utilization profiling during inference
- Entity-level precision / recall / F1 evaluation against ground-truth labels
- MLflow experiment tracking for cross-model comparison

**Dataset:** [gretelai/synthetic_pii_finance_multilingual](https://huggingface.co/datasets/gretelai/synthetic_pii_finance_multilingual) (English subset).

**Infra:** Single-node A100 80 GB (`Standard_NC24ads_A100_v4`), ML Runtime 16.4 LTS GPU.

## 1. Parameters

In [ ]:
dbutils.widgets.text("model_id", "Qwen/Qwen3-4B", "HuggingFace Model ID")
dbutils.widgets.text("num_rows", "500", "Number of PII documents")
dbutils.widgets.text("max_tokens", "500", "Max output tokens")
dbutils.widgets.text("output_catalog", "shm", "Unity Catalog")
dbutils.widgets.text("output_schema", "default", "Schema")

## 2. Install dependencies

Pins are validated against ML Runtime 16.4 LTS (PyTorch 2.5, CUDA 12.4). FlashInfer provides optimized sampling kernels for vLLM.

In [ ]:
import subprocess, sys, torch, numpy as np, os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

torch_ver = torch.__version__
np_ver = np.__version__
cuda_tag = torch.version.cuda.replace(".", "")
torch_short = ".".join(torch_ver.split(".")[:2])

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    f"torch=={torch_ver}",
    f"numpy=={np_ver}",
    "vllm==0.8.5.post1",
    "transformers>=4.51.0,<5.0",
    "datasets",
    "pydantic>=2.0",
    "-q",
])

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "flashinfer-python",
    "-i", f"https://flashinfer.ai/whl/cu{cuda_tag}/torch{torch_short}/",
    "-q",
])
dbutils.library.restartPython()

## 3. Re-read parameters, configure logging, set MLflow experiment

`restartPython()` clears the interpreter so widgets must be re-read. Logs write to local disk (UC Volume mounts don't support `fsync`) and are copied to the volume at the end. MLflow tracks all metrics for cross-model comparison.

In [ ]:
import logging, json, mlflow
from pathlib import Path
from datetime import datetime

MODEL_ID = dbutils.widgets.get("model_id")
NUM_ROWS = int(dbutils.widgets.get("num_rows"))
MAX_TOKENS = int(dbutils.widgets.get("max_tokens"))
OUTPUT_CATALOG = dbutils.widgets.get("output_catalog")
OUTPUT_SCHEMA = dbutils.widgets.get("output_schema")

MODEL_SAFE = MODEL_ID.replace("/", "--")
RUN_TS = datetime.utcnow().strftime("%Y%m%dT%H%M%S")

LOCAL_LOG_DIR = Path("/tmp/pii_profiling_logs")
LOCAL_LOG_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_LOG_FILE = LOCAL_LOG_DIR / f"{MODEL_SAFE}_{RUN_TS}.log"

VOLUME_LOG_DIR = Path(f"/Volumes/{OUTPUT_CATALOG}/{OUTPUT_SCHEMA}/llm_profiling")
VOLUME_LOG_DIR.mkdir(parents=True, exist_ok=True)

log = logging.getLogger("pii_profiling")
log.setLevel(logging.INFO)
log.handlers.clear()
fh = logging.FileHandler(str(LOCAL_LOG_FILE))
fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
log.addHandler(fh)

_user = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
mlflow.set_experiment(f"/Users/{_user}/pii-detection-profiling")

log.info(f"model={MODEL_ID} rows={NUM_ROWS} max_tokens={MAX_TOKENS}")
log.info(f"local_log={LOCAL_LOG_FILE}")

## 4. Cache model weights via UC Volume

Downloads model weights from HuggingFace once, persists a snapshot on a UC Volume, and copies to local SSD on subsequent runs. Avoids re-downloading across cluster restarts.

In [ ]:
import shutil
from pathlib import Path
from huggingface_hub import model_info, snapshot_download

VOLUME_CACHE = Path("/Volumes/shm/default/huggingface")
LOCAL_MODEL_DIR = Path("/local_disk0/hf_models") / MODEL_SAFE

info = model_info(MODEL_ID)
revision = info.sha
log.info(f"model_revision={revision[:12]}")

volume_snap = VOLUME_CACHE / MODEL_SAFE / revision
marker = volume_snap / ".cache_complete"

if marker.exists():
    log.info(f"volume_cache=HIT revision={revision[:12]}")
    if not LOCAL_MODEL_DIR.exists():
        log.info("copying volume->local")
        shutil.copytree(str(volume_snap), str(LOCAL_MODEL_DIR))
else:
    log.info(f"volume_cache=MISS downloading from HuggingFace")
    snapshot_download(MODEL_ID, revision=revision, local_dir=str(LOCAL_MODEL_DIR))
    volume_snap.mkdir(parents=True, exist_ok=True)
    shutil.copytree(str(LOCAL_MODEL_DIR), str(volume_snap), dirs_exist_ok=True)
    marker.touch()
    log.info(f"persisted to {volume_snap}")

MODEL_PATH = str(LOCAL_MODEL_DIR)

## 5. Load model into GPU

Loads the model in bfloat16 with prefix caching enabled. `VLLM_ENABLE_V1_MULTIPROCESSING=0` is required to run inside a Databricks notebook kernel.

In [ ]:
import os, subprocess, torch

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

def gpu_mem_info():
    out = subprocess.check_output([
        "nvidia-smi", "--query-gpu=memory.used,memory.free,memory.total",
        "--format=csv,nounits,noheader",
    ]).decode().strip()
    return tuple(int(x) for x in out.split(","))

def gpu_utilization():
    out = subprocess.check_output([
        "nvidia-smi", "--query-gpu=utilization.gpu,utilization.memory",
        "--format=csv,nounits,noheader",
    ]).decode().strip()
    return tuple(int(x) for x in out.split(","))

pre_used, _, _ = gpu_mem_info()

from vllm import LLM

llm = LLM(
    model=MODEL_PATH,
    trust_remote_code=True,
    dtype="bfloat16",
    max_model_len=4096,
    max_num_seqs=256,
    enable_prefix_caching=True,
    gpu_memory_utilization=0.80,
    enforce_eager=True,
)

post_used, post_free, post_total = gpu_mem_info()
weight_mem_mb = post_used - pre_used
gpu_name = torch.cuda.get_device_name(0)

log.info(f"model_loaded={MODEL_ID} weight_mb={weight_mem_mb} gpu={gpu_name}")
log.info(f"gpu_mem used={post_used} free={post_free} total={post_total}")

## 6. Load PII dataset and extract ground truth

Streams the [gretelai/synthetic_pii_finance_multilingual](https://huggingface.co/datasets/gretelai/synthetic_pii_finance_multilingual) dataset, filters to English documents under 3 000 chars, and parses the `pii_spans` column into typed ground-truth entities for evaluation.

In [ ]:
from datasets import load_dataset
import pandas as pd
import json, ast

GRETEL_TO_PII_TYPE = {
    "name": "PERSON", "first_name": "PERSON", "last_name": "PERSON",
    "ssn": "SSN", "social_security_number": "SSN",
    "street_address": "ADDRESS", "address": "ADDRESS", "city": "ADDRESS",
    "state": "ADDRESS", "zip_code": "ADDRESS",
    "email": "EMAIL", "email_address": "EMAIL",
    "phone_number": "PHONE", "phone": "PHONE",
    "credit_card_number": "CREDIT_CARD", "credit_card": "CREDIT_CARD",
    "bank_routing_number": "BANK_ACCOUNT", "iban": "BANK_ACCOUNT",
    "account_number": "BANK_ACCOUNT", "bank_account": "BANK_ACCOUNT",
    "date_of_birth": "DATE_OF_BIRTH", "dob": "DATE_OF_BIRTH",
}

def parse_ground_truth(pii_spans_raw, input_text):
    """Extract ground truth PII entities from the dataset's pii_spans column."""
    try:
        if isinstance(pii_spans_raw, str):
            spans = json.loads(pii_spans_raw)
        elif isinstance(pii_spans_raw, list):
            spans = pii_spans_raw
        else:
            return []
    except (json.JSONDecodeError, ValueError):
        try:
            spans = ast.literal_eval(pii_spans_raw)
        except Exception:
            return []

    entities = []
    for span in spans:
        if isinstance(span, dict):
            label = span.get("label", span.get("type", "")).lower().strip()
            value = span.get("text", span.get("value", ""))
            if not value and "start" in span and "end" in span:
                value = input_text[span["start"]:span["end"]]
            if value and len(value.strip()) > 1:
                pii_type = GRETEL_TO_PII_TYPE.get(label, "OTHER")
                entities.append({"type": pii_type, "value": value.strip()})
    return entities

ds = load_dataset(
    "gretelai/synthetic_pii_finance_multilingual",
    split="train",
    streaming=True,
)

rows = []
for row in ds:
    if row["language"] == "English" and len(row["generated_text"]) < 3000:
        text = row["generated_text"]
        gt = parse_ground_truth(row["pii_spans"], text)
        rows.append({
            "doc_id": len(rows),
            "document_type": row["document_type"],
            "input_text": text,
            "pii_spans": row["pii_spans"],
            "ground_truth": gt,
            "gt_count": len(gt),
        })
    if len(rows) >= NUM_ROWS:
        break

pii_df = pd.DataFrame(rows)
log.info(f"dataset docs={len(pii_df)} doc_types={pii_df['document_type'].nunique()} "
         f"avg_chars={pii_df['input_text'].str.len().mean():.0f} "
         f"gt_entities={pii_df['gt_count'].sum()} avg_per_doc={pii_df['gt_count'].mean():.1f}")

## 7. PII output schema and system prompt

The LLM output schema contains only `has_pii` and `entities` -- no `masked_text`. Generating the masked document is done programmatically from the entity list after parsing, which keeps output tokens small and parse rates high. vLLM's guided decoding constrains generation to valid JSON matching this schema.

In [ ]:
from enum import Enum
from pydantic import BaseModel, Field
import json

class PIIType(str, Enum):
    PERSON = "PERSON"
    SSN = "SSN"
    ADDRESS = "ADDRESS"
    EMAIL = "EMAIL"
    PHONE = "PHONE"
    CREDIT_CARD = "CREDIT_CARD"
    BANK_ACCOUNT = "BANK_ACCOUNT"
    DATE_OF_BIRTH = "DATE_OF_BIRTH"
    OTHER = "OTHER"

class PIIEntity(BaseModel):
    entity_type: PIIType
    original_value: str = Field(description="Exact text span copied verbatim from the document")
    salt_id: str = Field(description="Deterministic placeholder, e.g. PERSON_001, SSN_001")

class PIIDetectionResult(BaseModel):
    has_pii: bool
    entities: list[PIIEntity]

pii_schema = PIIDetectionResult.model_json_schema()
log.info(f"pii_schema fields={list(pii_schema.get('properties', {}).keys())}")

In [ ]:
from transformers import AutoTokenizer

SYSTEM_PROMPT = """\
You are a PII detection engine. Given a document, extract every personally \
identifiable information (PII) entity. Return valid JSON only.

## PII categories

PERSON -- Full names ("John Smith"), first names ("Sarah"), last names \
("Williams"), titles with names ("Dr. Maria Garcia", "Mr. James Lee"). \
Report each distinct person as one entity; combine first + last when adjacent.

SSN -- Social Security Numbers in any format: "123-45-6789", "123 45 6789", \
"123456789".

ADDRESS -- Street addresses, city/state/ZIP, or full mailing addresses. \
Combine contiguous components into one entity: "742 Evergreen Terrace, \
Springfield, IL 62704". Individual city names or ZIP codes that appear alone \
are also ADDRESS entities.

EMAIL -- Email addresses: "user@example.com".

PHONE -- Phone numbers in any format including country codes: \
"(555) 123-4567", "+1-800-555-0199", "555.123.4567".

CREDIT_CARD -- Credit/debit card numbers: "4111-1111-1111-1111", \
"5500 0000 0000 0004".

BANK_ACCOUNT -- Bank account numbers, routing numbers, IBANs, SWIFT/BIC \
codes: "Account #123456789", "Routing: 021000021", \
"IBAN: DE89370400440532013000".

DATE_OF_BIRTH -- Dates of birth in any format: "01/15/1990", \
"March 3, 1985", "1990-01-15", "born on May 12, 1978".

OTHER -- Any PII not covered above (passport numbers, driver's license \
numbers, tax IDs, national ID numbers).

## Rules

1. Extract EVERY occurrence. Missing an entity is worse than a false positive.
2. `original_value` must be the EXACT text span from the document -- copy \
verbatim, do not reformat or paraphrase.
3. Number salt_ids sequentially within each type: PERSON_001, PERSON_002, \
SSN_001, etc.
4. If the same value appears multiple times, reuse its salt_id.
5. For compound addresses (street + city + state + ZIP on the same line or \
sentence), report as one ADDRESS entity.
6. First name and last name that appear together are one PERSON entity; if \
only a first or last name appears alone elsewhere, report it separately.
7. Partial card/account numbers ("ending in 4532") are still BANK_ACCOUNT \
or CREDIT_CARD entities.

## Example 1

Input:
Dear Mr. James Wilson, your account ending in 4532 at 742 Evergreen \
Terrace, Springfield, IL 62704 has been flagged. Please contact us at \
support@finbank.com or call (312) 555-0198. SSN on file: 287-65-4321. \
Date of birth: 04/12/1983.

Output:
{"has_pii": true, "entities": [\
{"entity_type": "PERSON", "original_value": "James Wilson", \
"salt_id": "PERSON_001"}, \
{"entity_type": "BANK_ACCOUNT", "original_value": "4532", \
"salt_id": "BANK_ACCOUNT_001"}, \
{"entity_type": "ADDRESS", "original_value": "742 Evergreen Terrace, \
Springfield, IL 62704", "salt_id": "ADDRESS_001"}, \
{"entity_type": "EMAIL", "original_value": "support@finbank.com", \
"salt_id": "EMAIL_001"}, \
{"entity_type": "PHONE", "original_value": "(312) 555-0198", \
"salt_id": "PHONE_001"}, \
{"entity_type": "SSN", "original_value": "287-65-4321", \
"salt_id": "SSN_001"}, \
{"entity_type": "DATE_OF_BIRTH", "original_value": "04/12/1983", \
"salt_id": "DATE_OF_BIRTH_001"}]}

## Example 2

Input:
ACCOUNT STATEMENT -- Prepared for Elena Rodriguez (DOB: 1990-06-22). \
Mailing address: 1800 K Street NW, Washington, DC 20006. \
Primary phone: +1-202-555-0147. Email: e.rodriguez@globalmail.net. \
Routing number 091000019, account 00112233445. \
Visa ending 8821. SSN: 601-33-8877.

Output:
{"has_pii": true, "entities": [\
{"entity_type": "PERSON", "original_value": "Elena Rodriguez", \
"salt_id": "PERSON_001"}, \
{"entity_type": "DATE_OF_BIRTH", "original_value": "1990-06-22", \
"salt_id": "DATE_OF_BIRTH_001"}, \
{"entity_type": "ADDRESS", "original_value": "1800 K Street NW, \
Washington, DC 20006", "salt_id": "ADDRESS_001"}, \
{"entity_type": "PHONE", "original_value": "+1-202-555-0147", \
"salt_id": "PHONE_001"}, \
{"entity_type": "EMAIL", "original_value": "e.rodriguez@globalmail.net", \
"salt_id": "EMAIL_001"}, \
{"entity_type": "BANK_ACCOUNT", "original_value": "091000019", \
"salt_id": "BANK_ACCOUNT_001"}, \
{"entity_type": "BANK_ACCOUNT", "original_value": "00112233445", \
"salt_id": "BANK_ACCOUNT_002"}, \
{"entity_type": "CREDIT_CARD", "original_value": "8821", \
"salt_id": "CREDIT_CARD_001"}, \
{"entity_type": "SSN", "original_value": "601-33-8877", \
"salt_id": "SSN_001"}]}"""

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

system_tokens = tokenizer.encode(SYSTEM_PROMPT)
log.info(f"system_prompt_tokens={len(system_tokens)}")

## 8. Build prompts with shared system prefix

Every prompt starts with the identical system message. vLLM's prefix caching computes those KV blocks once and reuses them across the batch.

In [ ]:
formatted_prompts = []
for _, row in pii_df.iterrows():
    text = row["input_text"]
    if len(text) > 2000:
        text = text[:2000]
    prompt = tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user", "content": f"Extract all PII entities from this document:\n\n{text}"}],
        tokenize=False,
        add_generation_prompt=True,
    )
    formatted_prompts.append(prompt)

log.info(f"prompts_prepared={len(formatted_prompts)} prefix_cached=True")

## 9. Batched inference with GPU profiling

Splits prompts into batches of 50 and captures a GPU snapshot (memory, utilization) between each batch. This makes memory leaks visible as a monotonic increase in `used_mb` across batches. A background thread also samples at 2 Hz for continuous coverage.

In [ ]:
import time, threading, gc
from vllm import SamplingParams
from vllm.sampling_params import GuidedDecodingParams

BATCH_SIZE = 50

guided = GuidedDecodingParams(json=pii_schema)
sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.1,
    guided_decoding=guided,
)

class GPUMonitor:
    def __init__(self, interval=0.5):
        self.interval = interval
        self.samples = []
        self._stop = threading.Event()

    def _poll(self):
        while not self._stop.is_set():
            try:
                gpu_pct, mem_pct = gpu_utilization()
                used, free, total = gpu_mem_info()
                self.samples.append({"t": time.perf_counter(), "gpu_pct": gpu_pct,
                                     "mem_pct": mem_pct, "used_mb": used})
            except Exception:
                pass
            self._stop.wait(self.interval)

    def start(self):
        self.samples.clear()
        self._stop.clear()
        self._thread = threading.Thread(target=self._poll, daemon=True)
        self._thread.start()
        return self

    def stop(self):
        self._stop.set()
        self._thread.join(timeout=2)
        return self.samples

raw_outputs = []
batch_profiles = []
_, _, total_mem = gpu_mem_info()

monitor = GPUMonitor(interval=0.5).start()
t_start = time.perf_counter()

for batch_idx in range(0, len(formatted_prompts), BATCH_SIZE):
    batch = formatted_prompts[batch_idx : batch_idx + BATCH_SIZE]
    batch_num = batch_idx // BATCH_SIZE

    pre_used, pre_free, _ = gpu_mem_info()
    pre_gpu, _ = gpu_utilization()

    t0 = time.perf_counter()
    batch_out = llm.generate(batch, sampling_params=sampling_params)
    t1 = time.perf_counter()

    post_used, post_free, _ = gpu_mem_info()
    post_gpu, _ = gpu_utilization()

    batch_tokens = sum(len(o.outputs[0].token_ids) for o in batch_out)
    batch_tps = batch_tokens / (t1 - t0) if (t1 - t0) > 0 else 0

    profile = {
        "batch": batch_num,
        "docs": len(batch),
        "docs_cumulative": batch_idx + len(batch),
        "tokens": batch_tokens,
        "wall_sec": round(t1 - t0, 2),
        "tokens_per_sec": round(batch_tps, 1),
        "pre_gpu_mem_mb": pre_used,
        "post_gpu_mem_mb": post_used,
        "mem_delta_mb": post_used - pre_used,
        "pre_gpu_util": pre_gpu,
        "post_gpu_util": post_gpu,
    }
    batch_profiles.append(profile)
    log.info(f"batch={batch_num} docs={batch_idx+len(batch)}/{len(formatted_prompts)} "
             f"tok/s={batch_tps:.0f} gpu_mem={post_used}/{total_mem}MB "
             f"delta={post_used - pre_used:+d}MB")

    raw_outputs.extend((o.outputs[0].text, len(o.outputs[0].token_ids)) for o in batch_out)
    del batch_out

elapsed = time.perf_counter() - t_start
gpu_samples = monitor.stop()

total_output_tokens = sum(t for _, t in raw_outputs)
tokens_per_sec = total_output_tokens / elapsed if elapsed > 0 else 0

avg_gpu = sum(s["gpu_pct"] for s in gpu_samples) / len(gpu_samples) if gpu_samples else 0
peak_gpu = max((s["gpu_pct"] for s in gpu_samples), default=0)
peak_mem = max((s["used_mb"] for s in gpu_samples), default=0)

log.info(f"inference total docs={len(raw_outputs)} output_tokens={total_output_tokens} "
         f"wall_sec={elapsed:.2f} tokens_per_sec={tokens_per_sec:.1f}")
log.info(f"gpu avg_util={avg_gpu:.0f}% peak_util={peak_gpu}% peak_mem={peak_mem}MB")

mem_start = batch_profiles[0]["pre_gpu_mem_mb"]
mem_end = batch_profiles[-1]["post_gpu_mem_mb"]
log.info(f"gpu_mem first_batch={mem_start}MB last_batch={mem_end}MB drift={mem_end - mem_start:+d}MB")

del llm
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
log.info("gpu memory released")

## 10. Parse structured results

Validates each raw output against the Pydantic schema. Failed parses are recorded so the parse-success rate is captured alongside quality metrics.

In [ ]:
def build_masked_text(input_text, entities):
    """Replace each entity's original_value with its salt_id, longest-first to avoid partial overlaps."""
    masked = input_text
    for e in sorted(entities, key=lambda x: len(x.original_value), reverse=True):
        masked = masked.replace(e.original_value, e.salt_id)
    return masked

results = []
parse_errors = 0
all_predicted = []
all_ground_truth = []

for i, ((raw_text, n_tokens), (_, row)) in enumerate(zip(raw_outputs, pii_df.iterrows())):
    gt_entities = row["ground_truth"]

    try:
        parsed = PIIDetectionResult.model_validate_json(raw_text)
        pred_entities = [{"type": e.entity_type.value, "value": e.original_value}
                         for e in parsed.entities]
        entities_json = json.dumps([e.model_dump() for e in parsed.entities])
        masked = build_masked_text(row["input_text"], parsed.entities)
        results.append({
            "doc_id": row["doc_id"],
            "model_id": MODEL_ID,
            "document_type": row["document_type"],
            "input_text": row["input_text"],
            "masked_text": masked,
            "has_pii": parsed.has_pii,
            "num_entities": len(parsed.entities),
            "entities_json": entities_json,
            "gt_count": len(gt_entities),
            "output_tokens": n_tokens,
            "parse_ok": True,
        })
        all_predicted.append(pred_entities)
    except Exception:
        parse_errors += 1
        results.append({
            "doc_id": row["doc_id"],
            "model_id": MODEL_ID,
            "document_type": row["document_type"],
            "input_text": row["input_text"],
            "masked_text": row["input_text"],
            "has_pii": False,
            "num_entities": 0,
            "entities_json": "[]",
            "gt_count": len(gt_entities),
            "output_tokens": n_tokens,
            "parse_ok": False,
        })
        all_predicted.append([])
    all_ground_truth.append(gt_entities)

del raw_outputs

results_df = pd.DataFrame(results)
ok = results_df["parse_ok"].sum()
log.info(f"parse ok={ok}/{len(results_df)} errors={parse_errors} "
         f"detected={results_df['num_entities'].sum()} gt={results_df['gt_count'].sum()}")

## 11. Sample masked output

In [ ]:
good = results_df[results_df["parse_ok"] & (results_df["num_entities"] > 0)]
if len(good) > 0:
    sample = good.iloc[0]
    entities = json.loads(sample["entities_json"])
    log.info(f"sample doc_id={sample['doc_id']} type={sample['document_type']} "
             f"entities={len(entities)}")
    for e in entities[:5]:
        log.info(f"  salt={e['salt_id']} value={e['original_value'][:40]} type={e['entity_type']}")
else:
    log.warning("no successfully parsed documents with PII entities")

## 12. Evaluation -- Precision, Recall, F1

Entity-level matching: a predicted entity is a true positive if its normalized value is a substring of (or contains) a ground-truth entity of the same PII type. Unmatched predictions are false positives; unmatched ground-truth entities are false negatives. Both micro (corpus-level) and macro (document-averaged) aggregates are computed.

In [ ]:
def normalize(s):
    return s.lower().strip().replace("-", "").replace(" ", "")

def score_document(predicted, ground_truth):
    """Entity-level precision/recall/F1 for one document."""
    if not ground_truth and not predicted:
        return {"tp": 0, "fp": 0, "fn": 0, "precision": 1.0, "recall": 1.0, "f1": 1.0}

    gt_matched = [False] * len(ground_truth)
    tp = 0
    fp = 0

    for pred in predicted:
        pred_norm = normalize(pred["value"])
        matched = False
        for j, gt in enumerate(ground_truth):
            if gt_matched[j]:
                continue
            gt_norm = normalize(gt["value"])
            type_match = pred["type"] == gt["type"] or gt["type"] == "OTHER"
            value_match = (pred_norm in gt_norm or gt_norm in pred_norm
                           or pred_norm == gt_norm)
            if value_match and type_match:
                gt_matched[j] = True
                tp += 1
                matched = True
                break
        if not matched:
            fp += 1

    fn = sum(1 for m in gt_matched if not m)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {"tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall, "f1": f1}

doc_scores = []
for pred, gt in zip(all_predicted, all_ground_truth):
    doc_scores.append(score_document(pred, gt))

scores_df = pd.DataFrame(doc_scores)
results_df["precision"] = scores_df["precision"].values
results_df["recall"] = scores_df["recall"].values
results_df["f1"] = scores_df["f1"].values

total_tp = scores_df["tp"].sum()
total_fp = scores_df["fp"].sum()
total_fn = scores_df["fn"].sum()

micro_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
micro_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
micro_f1 = (2 * micro_precision * micro_recall / (micro_precision + micro_recall)
            if (micro_precision + micro_recall) > 0 else 0.0)

macro_precision = scores_df["precision"].mean()
macro_recall = scores_df["recall"].mean()
macro_f1 = scores_df["f1"].mean()

log.info(f"eval micro precision={micro_precision:.3f} recall={micro_recall:.3f} f1={micro_f1:.3f}")
log.info(f"eval macro precision={macro_precision:.3f} recall={macro_recall:.3f} f1={macro_f1:.3f}")
log.info(f"eval tp={total_tp} fp={total_fp} fn={total_fn}")

## 13. Log to MLflow

Logs params, metrics, and artifacts to an MLflow experiment run. Each model gets its own run so the MLflow UI provides side-by-side comparison across the sweep. The per-document results CSV and text log are attached as artifacts.

In [ ]:
import shutil

results_csv = LOCAL_LOG_DIR / f"{MODEL_SAFE}_{RUN_TS}_results.csv"
write_cols = [c for c in results_df.columns if c != "ground_truth"]
results_df[write_cols].to_csv(str(results_csv), index=False)

gpu_profile_csv = LOCAL_LOG_DIR / f"{MODEL_SAFE}_{RUN_TS}_gpu_profile.csv"
pd.DataFrame(batch_profiles).to_csv(str(gpu_profile_csv), index=False)

fh.flush()

with mlflow.start_run(run_name=MODEL_SAFE):
    mlflow.log_params({
        "model_id": MODEL_ID,
        "num_rows": NUM_ROWS,
        "max_tokens": MAX_TOKENS,
        "batch_size": BATCH_SIZE,
        "gpu": gpu_name,
        "gpu_total_mb": total_mem,
        "gpu_memory_utilization": 0.80,
        "max_model_len": 4096,
        "max_num_seqs": 256,
        "enforce_eager": True,
        "system_prompt_tokens": len(system_tokens),
    })
    mlflow.log_metrics({
        "tokens_per_sec": round(tokens_per_sec, 1),
        "wall_time_sec": round(elapsed, 2),
        "total_output_tokens": total_output_tokens,
        "weight_mem_mb": weight_mem_mb,
        "peak_gpu_mem_mb": peak_mem,
        "avg_gpu_util_pct": round(avg_gpu, 1),
        "peak_gpu_util_pct": peak_gpu,
        "gpu_mem_drift_mb": mem_end - mem_start,
        "parse_ok": int(ok),
        "parse_errors": parse_errors,
        "detected_entities": int(results_df["num_entities"].sum()),
        "gt_entities": int(results_df["gt_count"].sum()),
        "tp": int(total_tp),
        "fp": int(total_fp),
        "fn": int(total_fn),
        "micro_precision": round(micro_precision, 4),
        "micro_recall": round(micro_recall, 4),
        "micro_f1": round(micro_f1, 4),
        "macro_precision": round(macro_precision, 4),
        "macro_recall": round(macro_recall, 4),
        "macro_f1": round(macro_f1, 4),
    })
    for bp in batch_profiles:
        mlflow.log_metrics({
            "batch_gpu_mem_mb": bp["post_gpu_mem_mb"],
            "batch_tokens_per_sec": bp["tokens_per_sec"],
            "batch_gpu_util": bp["post_gpu_util"],
        }, step=bp["docs_cumulative"])
    mlflow.log_artifact(str(LOCAL_LOG_FILE))
    mlflow.log_artifact(str(results_csv))
    mlflow.log_artifact(str(gpu_profile_csv))
    run_id = mlflow.active_run().info.run_id

log.info(f"mlflow run_id={run_id}")

shutil.copy2(str(LOCAL_LOG_FILE), str(VOLUME_LOG_DIR / LOCAL_LOG_FILE.name))
log.info(f"log copied to {VOLUME_LOG_DIR}")

## 14. Exit summary

Notebook exit value returned to the Databricks job driver for quick status checks without opening the notebook run.

In [ ]:
exit_msg = json.dumps({
    "model_id": MODEL_ID,
    "tokens_per_sec": round(tokens_per_sec, 1),
    "micro_f1": round(micro_f1, 4),
    "parse_ok": int(ok),
    "parse_total": len(results_df),
})
log.info(f"run complete: {exit_msg}")
dbutils.notebook.exit(exit_msg)